# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 15 · Passing-line ablation across later games

Compare the preserved 72-column control with **control + pass-axis state** and **control + pass-axis state/history**. The failed direct-state bundle is absent. The model, folds, targets, rows, and training exposure are fixed. Maximum eight new coordinate fits, four per later fold. These game splits are reused; this is not independent confirmation.

In [ ]:
from pathlib import Path
import json, sys, subprocess, signal
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round6')
OUT = Path('/home/sagemaker-user/nfl-feature-round6-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Open the existing NFL space and extract Round 6 first.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
def run(stage, fold=2):
    cmd = [str(PY), str(KIT/'run_round.py'), stage, '--fold', str(fold)]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
        code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        process.wait(timeout=10)
        raise
    if code:
        raise RuntimeError(f'{stage} stopped ({code}). Preserve checkpoints; export the report. Do not retry an unchanged failure.')
def show(fig, name):
    visuals.save(fig, OUT, name).show()


## Fit fold 2
This makes four new coordinate fits at most. Control predictions are replayed, not refitted. Expected: `pass_axis_fold_complete`. Inspect the printed contrasts before continuing.

In [ ]:
run('fit', fold=2)
r = json.loads((OUT/'fold_2/summary.json').read_text())
print(json.dumps({'metrics': r['metrics'], 'contrasts': r['contrasts']}, indent=2))

## Conditional fold 3
If **both** candidate arms are at least 5% worse than control on fold 2, this command records `stopped_for_futility` and does no fitting. Otherwise both arms proceed unchanged. Do not modify settings or bypass the guard.

In [ ]:
run('fit', fold=3)

## Fresh-process no-refit replay
Reload completed weights and reproduce saved predictions exactly. Incomplete models are a stop, never silently fitted. Expected: `pass_axis_replay_exact`, `new_coordinate_models: 0`.

In [ ]:
run('replay')
print((OUT/'replay.json').read_text())

## Review every planned comparison
The acceptance rule uses both later-fold directions, >=1% pooled RMSE improvement, and the adjusted upper game-bootstrap bound below zero. History must improve on **both** the terminal-only treatment and control. No automatic promotion or feature selection uses these charts.

In [ ]:
show(visuals.fold_metrics(OUT), 'round6_folds')
show(visuals.contrasts(OUT), 'round6_contrasts')
show(visuals.horizon(OUT), 'round6_horizons')
show(visuals.retained(OUT), 'retained_columns')
print((OUT/'summary.json').read_text())

## Export and stop
Download the small report below. It contains aggregate results and execution receipts, not raw tracking, target arrays, predictions, or models. Save both notebooks; stop the existing JupyterLab space when done, without deleting it.

In [ ]:
run('report')
print(OUT/'nfl_feature_round6_report.zip')